# Module 2: GPTQ — Why INT4 Needs Smart Quantization

This notebook demonstrates why naive INT4 quantization fails and how GPTQ fixes it.

**What you'll learn:**
- Why naive INT4 rounding causes much worse quality than INT8
- What the Hessian matrix is and why it matters
- How GPTQ compensates for rounding errors
- The difference between weight-only and activation quantization

**Read first:** `docs/theory.md` section 7

**Time:** ~45 minutes

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

from src.quantization.absmax import absmax_quantize, absmax_dequantize
from src.quantization.gptq import gptq_quantize_layer, QuantizedLinear
from src.quantization.gptq_utils import compute_hessian, cholesky_inverse

print('Setup complete!')

## Part 1: Why Naive INT4 Is Bad

Let's demonstrate the quality cliff with a concrete example.

In [ ]:
# Create a simple linear layer and quantize it at different precisions
torch.manual_seed(42)
W = torch.randn(64, 64) * 0.1  # typical weight magnitude
x = torch.randn(16, 64) * 0.5  # typical activation

# Ground truth output
y_true = x @ W.T

# INT8 quantized
q8, s8 = absmax_quantize(W, bits=8)
W_int8 = absmax_dequantize(q8, s8)
y_int8 = x @ W_int8.T

# INT4 quantized (naive rounding)
q4, s4 = absmax_quantize(W, bits=4)
W_int4 = absmax_dequantize(q4, s4)
y_int4 = x @ W_int4.T

err_int8 = (y_true - y_int8).abs().mean().item()
err_int4 = (y_true - y_int4).abs().mean().item()

print('Output error (how wrong is the layer output?)')
print(f'  INT8 naive: {err_int8:.6f}')
print(f'  INT4 naive: {err_int4:.6f}')
print(f'  INT4 is {err_int4/err_int8:.1f}x worse than INT8!')
print()
print('Weight error (how wrong are the stored weights?)')
err_W_int8 = (W - W_int8).abs().mean().item()
err_W_int4 = (W - W_int4).abs().mean().item()
print(f'  INT8 weight error: {err_W_int8:.6f}')
print(f'  INT4 weight error: {err_W_int4:.6f}')

## Part 2: The Hessian — Which Weights Matter Most?

The Hessian tells us: "if I change this weight, how much does the output change?"

In [ ]:
# Simulate collecting activations (X) during a calibration forward pass
torch.manual_seed(0)
n_tokens = 500
d_in = 64
X = torch.randn(d_in, n_tokens) * 0.5   # [d_in, n_tokens]

# Compute Hessian
H = compute_hessian(X, damp=0.01)
print(f'Hessian shape: {H.shape}  ({d_in} × {d_in})')
print(f'Hessian is symmetric: {torch.allclose(H, H.T, atol=1e-5)}')
print(f'Diagonal values (weight importance): min={H.diagonal().min():.4f}, max={H.diagonal().max():.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

im = ax1.imshow(H.numpy(), cmap='Blues')
ax1.set_title('Hessian Matrix H\n(brighter = more important interaction)')
plt.colorbar(im, ax=ax1)

ax2.bar(range(len(H.diagonal()[:32])), H.diagonal()[:32].numpy())
ax2.set_title('Hessian Diagonal (first 32 dims)\nH[i,i] = importance of weight dimension i')
ax2.set_xlabel('Input dimension i')
ax2.set_ylabel('H[i,i]')

plt.tight_layout()
plt.show()
print('Higher diagonal = this input dimension appears more often in activations')
print('= quantization error here affects output more')
print('= GPTQ should compensate more carefully for errors in these columns')

In [ ]:
# Compute inverse Hessian using Cholesky decomposition
H_inv = cholesky_inverse(H)
print(f'H_inv shape: {H_inv.shape}')

# Verify: H @ H_inv should be close to identity
identity_approx = H @ H_inv
eye = torch.eye(d_in)
diff = (identity_approx - eye).abs().max().item()
print(f'Max |H @ H_inv - I|: {diff:.8f}  (should be near 0)')

plt.figure(figsize=(6, 5))
plt.imshow(identity_approx[:16, :16].numpy(), cmap='RdBu', vmin=-0.1, vmax=1.1)
plt.title('H @ H_inv (top-left 16×16)\nShould look like identity matrix')
plt.colorbar()
plt.show()

## Part 3: GPTQ vs Naive INT4 — The Error Compensation in Action

In [ ]:
# Apply GPTQ and compare against naive INT4
torch.manual_seed(42)
d_out, d_in = 64, 64
W = torch.randn(d_out, d_in) * 0.1

# Generate realistic activation statistics for Hessian
X_calib = torch.randn(d_in, 1000) * 0.5
H = compute_hessian(X_calib, damp=0.01)
H_inv = cholesky_inverse(H)

# Naive INT4
q4_naive, s4 = absmax_quantize(W, bits=4)
W_naive = absmax_dequantize(q4_naive, s4)
err_naive = (W - W_naive).abs()

# GPTQ INT4
W_gptq_int, scales_gptq, _ = gptq_quantize_layer(W, H_inv, bits=4)
W_gptq = W_gptq_int.float() * scales_gptq.unsqueeze(0)
err_gptq = (W - W_gptq).abs()

print(f'Mean weight error:')
print(f'  Naive INT4: {err_naive.mean():.6f}')
print(f'  GPTQ INT4:  {err_gptq.mean():.6f}')
print(f'  GPTQ is {err_naive.mean()/err_gptq.mean():.2f}x better!')

# Output error (what actually matters for generation quality)
x_test = torch.randn(8, d_in) * 0.5
y_true = x_test @ W.T
y_naive = x_test @ W_naive.T
y_gptq = x_test @ W_gptq.T

print(f'\nMean output error:')
print(f'  Naive INT4: {(y_true - y_naive).abs().mean():.6f}')
print(f'  GPTQ INT4:  {(y_true - y_gptq).abs().mean():.6f}')

# Visualize error heatmaps
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
vmax = err_naive[:16, :16].max()

im0 = axes[0].imshow(W[:16, :16].numpy(), cmap='RdBu')
axes[0].set_title('Original Weights (16×16 slice)')
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(err_naive[:16, :16].numpy(), cmap='hot', vmax=vmax)
axes[1].set_title(f'Naive INT4 Error\nmean={err_naive.mean():.4f}')
plt.colorbar(im1, ax=axes[1])

im2 = axes[2].imshow(err_gptq[:16, :16].numpy(), cmap='hot', vmax=vmax)
axes[2].set_title(f'GPTQ INT4 Error\nmean={err_gptq.mean():.4f}')
plt.colorbar(im2, ax=axes[2])

plt.tight_layout()
plt.show()
print('Darker = less error. GPTQ should appear darker (less error)!')

## Part 4: QuantizedLinear — How INT4 Is Stored in Memory

In [ ]:
# Show the INT4 packing trick: 2 values per byte
in_f, out_f = 32, 16
q_linear = QuantizedLinear(in_f, out_f, bits=4)

W_int = torch.randint(-7, 8, (out_f, in_f), dtype=torch.int8)
scales = torch.rand(in_f) * 0.1
zeros = torch.zeros(in_f)
q_linear.pack(W_int, scales, zeros)

print(f'Weight matrix shape: [{out_f}, {in_f}]')
print(f'FP16 storage would need: {out_f * in_f * 2} bytes')
print(f'INT4 packed storage:     {q_linear.weight_packed.nelement()} bytes')
print(f'Compression: {out_f * in_f * 2 / q_linear.weight_packed.nelement():.1f}x vs FP16')
print()
print('How packing works:')
print('  Original INT4 values (just 4 values shown):', W_int[0, :4].tolist())
print('  These get stored as 2 bytes packed:')
W_shifted = (W_int[0, :4] + 8).tolist()  # shift to unsigned
print(f'  Shifted to unsigned [0,15]: {W_shifted}')
print(f'  Byte 0 = {W_shifted[0]} | ({W_shifted[1]} << 4) = {W_shifted[0] | (W_shifted[1] << 4)} = 0x{W_shifted[0] | (W_shifted[1] << 4):02X}')
print(f'  Byte 1 = {W_shifted[2]} | ({W_shifted[3]} << 4) = {W_shifted[2] | (W_shifted[3] << 4)} = 0x{W_shifted[2] | (W_shifted[3] << 4):02X}')

# Verify round-trip
W_recovered = q_linear.dequantize()
W_expected = W_int.float() * scales.unsqueeze(0)
err = (W_recovered - W_expected).abs().max()
print(f'\nPack → unpack round-trip error: {err:.6f} (should be near 0)')

## Summary

Key takeaways from this notebook:

1. **Naive INT4 is ~18x coarser than INT8** — rounding errors compound across layer outputs
2. **The Hessian H = 2XX^T** measures weight importance from real activation statistics
3. **GPTQ compensates** by adjusting unquantized weights after each column is quantized
4. **Cholesky is critical** for stable Hessian inversion (direct inversion is numerically explosive)
5. **QuantizedLinear packs** two INT4 values per byte — 4x smaller than FP32, 2x smaller than FP16

**Next:** `03_gguf_inspection.ipynb` — explore how llama.cpp stores these quantized models on disk